# Web Search

In [1]:
import structlog
import time
import warnings
from dataclasses import dataclass
from datetime import datetime
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.tools import BaseTool
from langchain_core._api.beta_decorator import LangChainBetaWarning
from langchain_core.messages import AIMessage
from langchain_openrouter import ChatOpenRouter
from rich import pretty
from typing import cast
from world_cup_graph_agent.config.settings import Settings
from world_cup_graph_agent.observability.logging import configure_logging


warnings.filterwarnings("ignore", category=LangChainBetaWarning)
from langchain.mcp import MCPAdapter  # noqa: E402


pretty.install()
configure_logging()

In [2]:
def log(message: str) -> None:
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S:%f}] {message}")


async def invoke(
    logger: structlog.BoundLogger,
    model: ChatOpenRouter,
    prompt: str,
    tools: list[BaseTool] | None = None,
) -> dict:
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt="",
    )

    logger.info("Invoking...")
    start = time.perf_counter()

    result = await agent.ainvoke(
        {
            "messages": [{"role": "user", "content": prompt.strip()}],
        }
    )

    result["run_time"] = time.perf_counter() - start
    logger.info("Received")
    return result


async def invoke_with_exa(logger: structlog.BoundLogger, prompt: str, model) -> dict:
    start = time.perf_counter()
    async with MCPAdapter("https://mcp.exa.ai/mcp?defaultSearchType=fast") as exa:
        tools = await exa.list_tools()
        result = await invoke(logger, model, prompt, tools)
        result["run_time_with_mcp"] = time.perf_counter() - start
        return result


settings = Settings()


def get_model(providers: list[str]):
    return ChatOpenRouter(
        api_key=settings.openrouter_api_key,
        app_title=Settings.APP_NAME,
        model="openai/gpt-oss-120b",
        temperature=0.2,
        reasoning={"effort": "low"},
        openrouter_provider={
            "allow_fallbacks": False,
            "order": providers,
        },
    )


openrouter_web_tools = [
    {
        "type": "openrouter:web_search",
        "parameters": {
            "engine": "exa",
            "mode": "fast",
            "max_results": 5,
            "max_total_results": 15,
            "max_characters": 2_000,
        },
    },
    {
        "type": "openrouter:web_fetch",
        "parameters": {
            "engine": "exa",
            "max_uses": 3,
            "max_content_tokens": 8_000,
        },
    },
]

In [3]:
prompt = "Which new model was announced in OpenAI's DevDay September 2026?"

results: list[tuple[str, dict]] = []

In [4]:
results.append(
    (
        "exa-cerebras",
        await invoke_with_exa(
            prompt=prompt,
            logger=structlog.get_logger().bind(logger_name="exa-cerebras"),
            model=get_model(["cerebras/fp16"]),
        ),
    )
)

2026-09-29T22:52:31.873326Z [info] Invoking... [exa-cerebras]
2026-09-29T22:52:34.907516Z [info] Received [exa-cerebras]


In [5]:
results.append(
    (
        "exa-groq",
        await invoke_with_exa(
            prompt=prompt,
            logger=structlog.get_logger().bind(logger_name="exa-groq"),
            model=get_model(["groq"]),
        ),
    )
)

2026-09-29T22:52:36.427827Z [info] Invoking... [exa-groq]
2026-09-29T22:52:38.698142Z [info] Received [exa-groq]


In [6]:
results.append(
    (
        "exa-samba",
        await invoke_with_exa(
            prompt=prompt,
            logger=structlog.get_logger().bind(logger_name="exa-samba"),
            model=get_model(["sambanova"]),
        ),
    )
)

2026-09-29T22:52:40.228378Z [info] Invoking... [exa-samba]
2026-09-29T22:52:46.785459Z [info] Received [exa-samba]


In [7]:
results.append(
    (
        "openrouter-samba",
        await invoke(
            prompt=prompt,
            logger=structlog.get_logger().bind(logger_name="openrouter-samba"),
            # crusoe regularly throws TooManyRequestsResponseError
            model=cast(
                ChatOpenRouter,
                get_model(
                    ["sambanova"]
                    # ["deepinfra/turbo"]
                    # ["crusoe/bf16"]
                    # ["novita/fp4"]
                ).bind(
                    tools=openrouter_web_tools,
                ),
            ),
        ),
    )
)

2026-09-29T22:52:47.105239Z [info] Invoking... [openrouter-samba]
2026-09-29T22:52:51.249566Z [info] Received [openrouter-samba]


In [8]:
results.append(
    (
        "openrouter-deepinfra",
        await invoke(
            prompt=prompt,
            logger=structlog.get_logger().bind(logger_name="openrouter-deepinfra"),
            # crusoe regularly throws TooManyRequestsResponseError
            model=cast(
                ChatOpenRouter,
                get_model(
                    # ["sambanova"]
                    ["deepinfra/turbo"]
                    # ["crusoe/bf16"]
                    # ["novita/fp4"]
                ).bind(
                    tools=openrouter_web_tools,
                ),
            ),
        ),
    )
)

2026-09-29T22:52:51.288754Z [info] Invoking... [openrouter-deepinfra]
2026-09-29T22:52:57.125634Z [info] Received [openrouter-deepinfra]


In [9]:
@dataclass(frozen=True)
class ProviderUsage:
    input_tokens: int
    output_tokens: int
    total_tokens: int
    run_time: float
    run_time_with_mcp: float
    cost: float


def summarize_usage(result: dict) -> ProviderUsage:
    messages = result["messages"]
    run_time = result["run_time"]
    run_time_with_mcp = result.get("run_time_with_mcp", 0)
    ai_messages = [message for message in messages if isinstance(message, AIMessage)]
    return ProviderUsage(
        input_tokens=sum(
            (message.usage_metadata or {}).get("input_tokens", 0)
            for message in ai_messages
        ),
        output_tokens=sum(
            (message.usage_metadata or {}).get("output_tokens", 0)
            for message in ai_messages
        ),
        total_tokens=sum(
            (message.usage_metadata or {}).get("total_tokens", 0)
            for message in ai_messages
        ),
        run_time=run_time,
        run_time_with_mcp=run_time_with_mcp,
        cost=sum(
            float((message.response_metadata or {}).get("cost", 0) or 0)
            for message in ai_messages
        ),
    )


def render_summary_table(results: list[tuple[str, dict]]) -> str:
    lines = [
        "| Run | Input tks | Output tks | Invoke time | Total time | Cost | Response |",
        "|---|---|---|---|---|---|---|",
    ]
    for title, result in results:
        usage = summarize_usage(result)
        total_time = usage.run_time_with_mcp or usage.run_time
        response = " ".join(result["messages"][-1].content.split())
        response = response.replace("|", "\\|").replace("$", "\\$")
        lines.append(
            f"| {title} | {usage.input_tokens:,} | {usage.output_tokens:,} "
            f"| {usage.run_time:.2f} s | {total_time:.2f} s "
            f"| \\${usage.cost:.5f} | {response} |"
        )
    return "\n".join(lines)


display(Markdown(render_summary_table(results)))

| Run | Input tks | Output tks | Invoke time | Total time | Cost | Response |
|---|---|---|---|---|---|---|
| exa-cerebras | 2,609 | 125 | 3.02 s | 4.70 s | \$0.00101 | The new model announced at OpenAI’s **DevDay September 2026** was **GPT‑6.1 Sol**. |
| exa-groq | 2,694 | 201 | 2.27 s | 3.46 s | \$0.00052 | **The new model announced at OpenAI’s DevDay in September 2026 was **`GPT‑6.1 Sol`. During the keynote Sam Altman introduced GPT‑6.1 Sol as the latest “major upgrade” to the GPT‑6 family—positioned between the fast, inexpensive **GPT‑6 Luna** and the high‑performance **GPT‑6 Astra**. It was highlighted for strong performance on professional‑work tasks, coding, computer use, and agentic workloads, while being offered at a substantially lower cost than the Astra tier. |
| exa-samba | 2,686 | 121 | 6.56 s | 7.65 s | \$0.00049 | The new model announced at OpenAI’s DevDay in September 2026 was **GPT‑6.1 Sol**. |
| openrouter-samba | 2,014 | 136 | 4.14 s | 4.14 s | \$0.00741 | The new model announced at OpenAI’s **DevDay September 2026** was **GPT‑6.1 Sol**. It was presented as a major upgrade over the earlier GPT‑6 Sol model, positioned between the fast‑inexpensive “Luna” model and the higher‑performance “Astra” line, and targeted at professional‑grade coding, computer‑use, and agent‑centric workloads. |
| openrouter-deepinfra | 1,999 | 125 | 5.84 s | 5.84 s | \$0.00737 | The new model announced at OpenAI’s DevDay in September 2026 was **GPT‑6.1 Sol**. It was introduced as a “major upgrade” to the earlier GPT‑6 Sol, offering lower‑cost, higher‑performance capabilities for coding, computer‑use, and professional‑task workloads. |